# Task 7 — Packaging and Libraries

This task concerns Python packaging, dependencies, the difference between scripts and libraries, and command-line interfaces. All questions refer to the `allocations` package — see `pyproject.toml`, `src/allocations/`, and `src/allocations/cli.py`.

**Total marks: 25**

---

## Part A (5)

- (2) State what it means for a project to *depend on* another library, and what it means for a project to *be a dependency of* another library. Use the `allocations` package as the example for both senses, even if the latter is hypothetical.

- (1) Look at `pyproject.toml` in this repository. State **two** distinct pieces of information that this file records about the `allocations` package, beyond just its name.

- (2) Explain the difference between a *collection of scripts* and a *Python package / library*. Identify which of these `allocations` is, and cite one piece of evidence from the repository structure that supports your answer. 

### Answer — Part A
a project is said to *depend on* another library if it needs that library to work. That library ussually is ussually seen thhroough import, can also be something listed in pyporject.toml.

 a project to *be a dependency of* another library means that project must be is need for that the other code to run. i.e if A needs B, depends on B and B is a dependency of A. allocations.matrix depends on allocations.solvers.
allocations would be a dependency of another project if that project imported and used allocations.



pyproject.toml records 
1. the programming language the project was written in 
2. command line for the project API 


 If the project involves many files working in an integrated way on a shared purpose (i.e they depend on each other) this is a package. Crucially these files will be organised around a shared API for packages, and not so for collections of scripts. allocations is a package it's files are built around a shared API, the command line for the API can be found in the pyproject.toml file:
[project.scripts]
allocations = "allocations.cli:main"







[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "allocations"
version = "0.1.0"
description = "Assign CAT workers to jobs from .wkab files and write .catal allocations."
readme = "README.md"
requires-python = ">=3.10"
authors = [
    { name = "CAT" }
]
license = { text = "MIT" }
classifiers = [
    "Programming Language :: Python :: 3",
    "Programming Language :: Python :: 3 :: Only",
    "Programming Language :: Python :: 3.10",
    "Programming Language :: Python :: 3.11",
    "Programming Language :: Python :: 3.12",
]

[project.scripts]
allocations = "allocations.cli:main"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
pythonpath = ["src"]
testpaths = ["tests"]


In [ ]:
# Optional: inspect pyproject.toml
from pathlib import Path
# print(Path('../pyproject.toml').read_text())

## Part B (8)

Daedalus is considering replacing the hand-rolled Hungarian-algorithm implementation in `allocations.solvers` with `scipy.optimize.linear_sum_assignment` from SciPy.

- (3) Identify and explain three different things Daedalus should look at *about SciPy* before deciding whether to add it as a dependency. (Hint: think about quality of tests, documentation, maintenance, scope.)

- (2) State **two** practical drawbacks for users of `allocations` if SciPy becomes a hard dependency. For each drawback, suggest one mitigation.

- (3) The current `allocations` package only depends on the Python standard library — `pyproject.toml` declares no `dependencies`. Suppose Daedalus decided to make SciPy an *optional* dependency, used only when the user passes `solver="scipy-hungarian"`. Sketch (in words) how you would design this so that:
    1. `pip install allocations` continues to work without SciPy installed,
    2. `from allocations import allocate_workers` does not raise `ImportError` when SciPy is missing,
    3. attempting to use the `"scipy-hungarian"` solver without SciPy raises a clear and helpful error message.

### Answer — Part B

What dependencies does the SciPy itself have,Are there any tests fo it?,Are there many or few contributors.

It may would not work with later version of Scipy. Mitigation: Specify range of SciPy version in pyproject.toml and run CI tests against supported versions.

It may have problems with platform compatibility, SciPy may be dificult install in some operating system, beeter to keep a pure python falback solver.

In pyproject.toml do not put SciPy in dependencies, instead add as an optional dependency.Import SciPy  only inside  solvers that use it.raise error only if those solver are asked by the user.

## Part C (12)

Daedalus wants to add a new sub-command to the existing CLI in `src/allocations/cli.py`. The new sub-command should be invoked as:

```bash
allocations-summary <input.wkab> --metric cost
```

It should read the workers, run `allocate_workers`, and **print** a one-line summary in the form:

```
<solver> on <metric>: <N> workers assigned, total penalty <P>
```

It should **not** write any output file. It should accept the same `--solver` and `--requirements` flags as the existing CLI.

- (3) Identify what changes would be needed in `pyproject.toml` to expose this as a new console script entry point alongside the existing `allocations` script. Show the relevant section of `pyproject.toml` after your edit.

- (6) Implement the function `summary_main(argv=None)` for this CLI in the code cell below, using the `argparse` library. Re-use existing functionality from the `allocations` package wherever possible (e.g. `read_wkab`, `allocate_workers`). Your implementation should also reuse the `_parse_requirements` helper that already exists in `allocations.cli`.

- (3) Demonstrate `summary_main` by calling it with a small `argv` list pointing to `examples/cat.wkab`, with `--metric skill` and `--requirements 1,2,1`. Capture the printed output and show it.

### Answer — pyproject.toml change (C.1)

```toml
[project.scripts]
allocations = "allocations.cli:main"
allocations-summary = "allocations.cli:allocations-summary_main"

```

In [ ]:
import argparse
from pathlib import Path

from allocations import allocate_workers, read_wkab
from allocations.cli import _parse_requirements


def summary_main(argv=None):
    """Print a one-line allocation summary for an input .wkab file."""
    parser= argparse.ArgumentParser(
        prog="allocations-summary"
    )
    parser.add_argument("input", type=Path, help="Path to the input .wkab JSON file.")
    parser.add_argument(
            "--metric",
            choices=("cost"),
            help="Penalty metric to optimise.",
    )
    args = parser.parse_args(argv)

    try:
        workers = read_wkab(args.input)
        result = allocate_workers(
            workers,
            metric=args.metric,
            requirements=args.requirements,
            solver=args.solver,
        )
        allocate_workers(workers)
        


In [2]:
import argparse
from pathlib import Path

from allocations import allocate_workers, read_wkab
from allocations.cli import _parse_requirements


def summary_main(argv=None):
    """Print a one-line allocation summary for an input .wkab file."""
    parser = argparse.ArgumentParser(
        prog="allocations-summary",
        description="Print a one-line allocation summary.",
    )
    parser.add_argument("input", type=Path, help="Path to the input .wkab JSON file.")
    parser.add_argument(
        "--metric",
        choices=("cost", "skill", "time"),
        default="cost",
        help="Penalty metric to optimise.",
    )
    parser.add_argument(
        "--solver",
        choices=("hungarian", "exhaustive", "greedy", "greedy-reverse"),
        default="hungarian",
        help="Solver to use.",
    )
    parser.add_argument(
        "--requirements",
        "-r",
        type=_parse_requirements,
        help="Comma-separated non-negative counts for each original job, e.g. 1,2,1.",
    )

    args = parser.parse_args(argv)

    workers = read_wkab(args.input)
    result = allocate_workers(
        workers,
        metric=args.metric,
        requirements=args.requirements,
        solver=args.solver,
    )

    print(
        f"{result.solver} on {result.metric}: "
        f"{len(result.slot_assignment)} workers assigned, "
        f"total penalty {result.total_penalty:g}"
    )
    return 0


In [ ]:
summary_main(["../examples/cat.wkab", "--metric", "skill", "--requirements", "1.,2.,1."])

NameError: name 'summary_main' is not defined